In [1]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader
from analysis import fit_peak_expbg

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used

# Energy emission peaks in MeV
Cs137_MeV = 0.662
Na22_MeV = [0.511, 1.275, 1.786]
Co60_MeV = [1.173, 1.332, 2,505]
Am241_MeV = 0.0595
# Background emission
K40_MeV = 1.460
Tl208_MEV = 2.614

colors = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen, ROOT.kOrange, ROOT.kViolet, ROOT.kYellow, ROOT.kSpring, ROOT.kCyan,]

In [2]:
files = sorted((project_root/'data/260323').glob('SUBT_*'))
acqs = [SiphraAcquisition(file) for file in files]
print(acqs)

[SIPHRAAcquisition(File: 'SUBT_01_EnergyResolution_thr30_gain1over100_1over3_Background'), SIPHRAAcquisition(File: 'SUBT_02_EnergyResolution_thr30_gain1over100_1over3_Cs137'), SIPHRAAcquisition(File: 'SUBT_03_EnergyResolution_thr30_gain1over100_1over3_Background2'), SIPHRAAcquisition(File: 'SUBT_04_EnergyResolution_thr30_gain1over100_1over3_Na22'), SIPHRAAcquisition(File: 'SUBT_05_EnergyResolution_thr30_gain1over100_1over3_Background4'), SIPHRAAcquisition(File: 'SUBT_06_EnergyResolution_thr30_gain1over100_1over3_Co60'), SIPHRAAcquisition(File: 'SUBT_07_EnergyResolution_thr30_gain1over100_1over3_Background6'), SIPHRAAcquisition(File: 'SUBT_08_EnergyResolution_thr30_gain1over100_1over3_Am241'), SIPHRAAcquisition(File: 'SUBT_09_EnergyResolution_thr30_gain1over100_1over3_Background8')]


In [3]:
# histograms
hists = {}
sources = []
for sgnl, bg in zip(acqs[1::2], acqs[2::2]):
    filepath = sgnl.filepath.stem
    src = (MetadataLoader.load(sgnl.metadataFile)).source
    print(src)
    sources.append(src)
    # Signal and Background
    hist_sgnl = ROOT.TH1F(f"{src} Signal", "", BITS12, 0, BITS12)
    hist_bg = ROOT.TH1F(f"{src} Background", "", BITS12, 0 , BITS12)
    hist_sgnl.Fill(sgnl['s']/len(sgnl.active_chs))
    hist_bg.Fill(bg['s']/len(bg.active_chs))
    hist_bg.Scale(sgnl.exposure/bg.exposure)
    # Clean spectrum
    hist_clean = hist_sgnl.Clone(f"{src} Clean")
    hist_clean.Add(hist_bg, -1)
    for hist in [hist_sgnl, hist_bg, hist_clean]:
        # hist.SetTitle(r"^{22}Na spectra - CI gain = 1/0.75 pC")
        hist.GetXaxis().SetTitle("ADC channel number")
        hist.GetYaxis().SetTitle(r"Normalized counts")
    hists[src] = hist_sgnl
    hists[f"{src}_BG"] = hist_bg
    hists[f"{src}_clean"] = hist_clean
    del hist_sgnl, hist_bg

Cs-137
Na22
Co-60
Am-241


In [4]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

Yinit = 0.82 # For stat boxes

cv = ROOT.TCanvas('cv', 'cv', 1600, 1200)
cv.Divide(2,2)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgds = [ROOT.TLegend(0.48, 0.61, 0.75, 0.83) for _ in range(4)]

for idx, (src, lg) in enumerate(zip(sources, lgds)):   
    cv.cd(idx+1)
    
    sgnl = hists[src]
    bg = hists[src+'_BG']
    clean = hists[src+'_clean']
    
    lg.AddEntry(sgnl, "Signal")
    lg.AddEntry(bg, "Background")
    lg.AddEntry(clean, "Subtracted")
    sgnl.SetLineColor(colors[0])
    bg.SetLineColor(colors[1])
    clean.SetLineColor(colors[2])
    sgnl.SetTitle(src)
    sgnl.Draw("hist")
    bg.Draw("sames hist")
    clean.Draw("sames hist")
    ROOT.gPad.Update()
    for i, sp in enumerate([sgnl, bg, clean]):
        st = sp.FindObject("stats")
        # print(type(st))
        st.SetY1NDC(Yinit - i*0.08)
        st.SetY2NDC(0.1)
    lg.Draw()
    cv.cd(idx+1).SetLogy()
    cv.Draw()

In [5]:
from analysis import *

#Calibration based on the 3 peaks of the Na-22 spectrum


hist = hists['Na22_clean']
energy_ranges = [(150, 200), (320, 420), (450, 520)]
energies = Na22_MeV

c_fit = calibration_fit(hist, energy_ranges, energies)
print(c_fit)

[0.004280158214429103, -0.2655932438428593]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =       50.431
NDf                       =           47
Edm                       =  6.26293e-06
NCalls                    =          110
Constant                  =      187.956   +/-   4.04943     
Mean                      =      177.384   +/-   1.67095     
Sigma                     =      37.5292   +/-   4.35406      	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      152.171
NDf                       =           97
Edm                       =  2.80563e-08
NCalls                    =           96
Constant                  =      606.675   +/-   4.80363     
Mean                      =       371.93   +/-   0.452134    
Sigma                     =      45.3973   +/-   0.782852     	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2               

In [8]:
#Calibrating Cs137 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[2], BITS12)
hist_cal_bg_Cs137.Scale(1/acqs[2].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[1], BITS12)
hist_cal_src_Cs137.Scale(1/acqs[1].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)


# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1600, 800)
cv1.Divide(2,1)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv1.cd(1)
hist_cal_bg_Cs137.SetLineColor(colors[0])
hist_cal_src_Cs137.SetLineColor(colors[1])
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"Signal ^{137}Cs", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
lg1.Draw()
cv1.cd(1).SetLogy()
#hist_cal_bg.GetYaxis().SetRangeUser(0,1e4)


lg2 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv1.cd(2)
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_clean_Cs137.Draw("hist")
cv1.cd(2).SetLogy()

cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, [(0.11, 0.85)], [Cs137_MeV])
print(res_Cs137)



[0.4476679045176995]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      297.546
NDf                       =          170
Edm                       =  2.83736e-09
NCalls                    =           66
Constant                  =      25.7351   +/-   0.0778773   
Mean                      =      0.52178   +/-   0.00031843  
Sigma                     =     0.125841   +/-   0.000252392  	 (limited)


Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [10]:
#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Na22 = calibrated_histogram(c_fit, acqs[4], BITS12)
hist_cal_bg_Na22.Scale(1/acqs[4].exposure) 
hist_cal_src_Na22 = calibrated_histogram(c_fit, acqs[3], BITS12)
hist_cal_src_Na22.Scale(1/acqs[3].exposure) 
hist_cal_clean_Na22 = hist_cal_src_Na22.Clone("Calibrated signal no BG")
hist_cal_clean_Na22.Add(hist_cal_bg_Na22, -1)


# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv2'):
    ROOT.gROOT.FindObject('cv2').Close()
cv2 = ROOT.TCanvas("cv2", "cv2", 1600, 800)
cv2.Divide(2,1)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv2.cd(1)
hist_cal_bg_Na22.SetLineColor(colors[0])
hist_cal_src_Na22.SetLineColor(colors[1])
lg1.AddEntry(hist_cal_bg_Na22, "Background", "l")
lg1.AddEntry(hist_cal_src_Na22, r"Signal ^{22}Na", "l")
hist_cal_src_Na22.Draw("hist")
hist_cal_bg_Na22.Draw("sames hist")
lg1.Draw()
cv2.cd(1).SetLogy()
#hist_cal_bg.GetYaxis().SetRangeUser(0,1e4)


lg2 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv2.cd(2)
hist_cal_clean_Na22.SetLineColor(colors[2])
hist_cal_clean_Na22.Draw("hist")
cv2.cd(2).SetLogy()

cv2.Draw()

#Calculate energy resolution
res_Na22 = energy_resolution(hist_cal_clean_Na22, [(0.13, 0.6), (1.1, 1.5), (1.5 , 2.1)],  Na22_MeV)
print(res_Na22)



[0.6098052497160661, 0.3310662458402524, 0.19985352745576823]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      114.793
NDf                       =          107
Edm                       =  1.05832e-07
NCalls                    =           78
Constant                  =     0.657598   +/-   0.0098944   
Mean                      =     0.487964   +/-   0.00391232  
Sigma                     =     0.132319   +/-   0.00277695   	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      288.638
NDf                       =           91
Edm                       =  1.06692e-07
NCalls                    =           90
Constant                  =      2.06376   +/-   0.0165861   
Mean                      =      1.31918   +/-   0.00187749  
Sigma                     =      0.17924   +/-   0.0030605    	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
C

Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
